# Мин. здравоохранения г.Москвы.

In [ ]:
# Импортируем библиотеки
import pandas as pd
import sqlite3

## Задание

- Найти пациентов стоящих на учете в МГКР (Московский городской канцер-регистр) с диагнозом C61 и получавших впервые Абиратерон в 2023 или 2024 году.
- Дополнить реестр информацией, у кого из этих пациентов есть приемы с подтверждённым диагнозом E11(включая значения после точки).

In [ ]:
# Загружаем данные
MGKR = pd.read_csv('C:/Users/Downloads/MGKR_reestr.csv')
RECEPTION = pd.read_csv('C:/Users/Downloads/RECEPTION_reestr.csv')
LLO = pd.read_csv('C:/Users/Download/LLO_reestr.csv')

In [ ]:
# Создаём подключение
conn = sqlite3.connect(':memory:')
MGKR.to_sql('MGKR_reestr', conn, index=False, if_exists='replace')
RECEPTION.to_sql('RECEPTION_reestr', conn, index=False, if_exists='replace')
LLO.to_sql('LLO_reestr', conn, index=False, if_exists='replace')

# Финальный SQL-запрос для SQLite
query = """
WITH tmp_llo AS (
  SELECT
    l.PATIENT_ID,
    l.MU_ID,
    l.DRUG_NAME_MNN,
    date(l.START_DATE) as START_DATE,
    ROW_NUMBER() OVER (PARTITION BY l.PATIENT_ID, l.DRUG_NAME_MNN ORDER BY l.START_DATE) AS RN_F,
    ROW_NUMBER() OVER (PARTITION BY l.PATIENT_ID, l.DRUG_NAME_MNN ORDER BY l.START_DATE DESC) AS RN_L
  FROM LLO_reestr l
  WHERE l.DRUG_NAME_MNN = 'Абиратерон' AND l.SALE_DATE IS NOT NULL
),
llo AS (
  SELECT
    s.PATIENT_ID,
    s.MU_ID,
    s.DRUG_NAME_MNN,
    s.START_DATE as FIRST_DATE,
    s2.START_DATE as LAST_DATE
  FROM tmp_llo s
  LEFT JOIN tmp_llo s2 ON s.PATIENT_ID = s2.PATIENT_ID AND s2.RN_L = 1
  WHERE s.RN_F = 1 AND strftime('%Y', s.START_DATE) IN ('2023', '2024')
),
fin_mgkr AS (
  SELECT
    m.patient_id,
    m.diag_code,
    m.diag_establish_date
  FROM MGKR_reestr m
  WHERE m.diag_code = 'C61'
    AND m.patient_id IN (SELECT DISTINCT PATIENT_ID FROM llo)
    AND m.patient_registration_status = 'Стоит на учете'
),
pat_E11 AS (
  SELECT
    dr.PATIENT_ID,
    dr.DIAGNOSIS_CODE,
    dr.DIAGNOSIS_STATUS,
    date(dr.EVENT_DATE) as EVENT_DATE,
    ROW_NUMBER() OVER (PARTITION BY dr.PATIENT_ID ORDER BY date(dr.EVENT_DATE) DESC) as rn
  FROM RECEPTION_reestr dr
  WHERE DIAGNOSIS_STATUS = 'подтвержден'
    AND dr.PATIENT_ID IN (SELECT DISTINCT PATIENT_ID FROM llo)
    AND dr.DIAGNOSIS_CODE BETWEEN 'E11' AND 'E11.9'
)
SELECT
  m.patient_id,
  m.diag_code,
  m.diag_establish_date,
  l.MU_ID,
  l.FIRST_DATE,
  l.LAST_DATE,
  CASE WHEN p.PATIENT_ID IS NOT NULL THEN 'Да' ELSE 'Нет' END as priem_E11,
  p.EVENT_DATE
FROM fin_mgkr m
JOIN llo l ON l.PATIENT_ID = m.patient_id
LEFT JOIN pat_E11 p ON p.PATIENT_ID = m.patient_id AND p.rn = 1
"""

result = pd.read_sql_query(query, conn)

print("\nРезультаты:")
display(result)

conn.close()


Результаты:


,patient_id,diag_code,diag_establish_date,MU_ID,FIRST_DATE,LAST_DATE,priem_E11,EVENT_DATE
0,16368892,C61,2021-12-01,11708903,2024-04-10,2024-08-30,Нет,None
1,20652645,C61,2024-09-06,10301578,2024-10-03,2025-06-24,Нет,None
2,20687206,C61,2009-01-26,11708903,2024-01-30,2024-04-29,Нет,None
3,20890714,C61,2024-09-06,10330308,2024-10-04,2025-06-19,Нет,None
4,20957663,C61,2023-06-21,11601278,2024-03-21,2024-04-15,Нет,None
...,...,...,...,...,...,...,...,...
404,26641722,C61,2022-10-24,11708903,2024-08-24,2025-06-11,Нет,None
405,23471820,C61,2018-01-29,11708903,2023-02-08,2023-11-18,Да,2024-03-15
406,23517885,C61,2022-02-25,11634078,2023-10-25,2024-03-26,Нет,None
407,23946074,C61,2021-07-02,10266728,2023-02-09,2025-06-05,Да,2023-12-13


#### Комментарий.
- Решил детальнее познакомиться с данными, поэтому, какие-то пункты в предобработке можно было-бы не далать. Оставил их для понимания хода размышлений.
- В sql-скрипте использовал простые конструкции для понимания хода решения + подробное комментирование и описание логики.

### Предварительная оценка качества данных

##### MGKR

In [ ]:
# Знакомимся
MGKR.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 199956 entries, 0 to 199955
Data columns (total 9 columns):
 #   Column                                Non-Null Count   Dtype 
---  ------                                --------------   ----- 
 0   patient_id                            199956 non-null  int64 
 1   patient_gender                        199956 non-null  object
 2   patient_registration_status           199956 non-null  object
 3   patient_registration_start_date       199937 non-null  object
 4   patient_registrations_removal_causes  31410 non-null   object
 5   patient_death_date                    28187 non-null   object
 6   diag_code                             199956 non-null  object
 7   diag_establish_date                   199956 non-null  object
 8   diag_stage                            167200 non-null  object
dtypes: int64(1), object(8)
memory usage: 13.7+ MB


In [ ]:
# Визуальный осмотр
display(MGKR)

,patient_id,patient_gender,patient_registration_status,patient_registration_start_date,patient_registrations_removal_causes,patient_death_date,diag_code,diag_establish_date,diag_stage
0,10591784,Женский,Стоит на учете,2022-09-26,NaN,NaN,C34.3,2022-07-05,I
1,10592025,Мужской,Снят с учета,2015-10-14,"умер от причин, связанных с основным заболеванием",2024-09-06,C34.9,2024-09-03,IV
2,10592025,Мужской,Снят с учета,2015-10-14,"умер от причин, связанных с основным заболеванием",2024-09-06,C85.1,2015-10-14,NaN
3,10592051,Мужской,Стоит на учете,2007-08-15,состоял по базалиоме,NaN,C61,2024-08-23,II
4,10592090,Женский,Снят с учета,1997-05-05,NaN,NaN,C73,1997-05-05,II
...,...,...,...,...,...,...,...,...,...
199951,23962490,Мужской,Снят с учета,2018-12-21,"умер от причин, связанных с основным заболеванием",2024-07-14,C61,2018-12-06,IV
199952,23962561,Мужской,Стоит на учете,2024-07-08,NaN,NaN,C61,2024-06-18,I
199953,23962583,Мужской,Стоит на учете,2023-01-31,NaN,NaN,C25.0,2023-01-09,IIB
199954,23962843,Мужской,Стоит на учете,2025-05-14,NaN,NaN,C61,2025-03-17,I


- Тип данных в столбцах `patient_registration_start_date`, `patient_death_date` и `diag_establish_date` не соответствуют содержимому, меняем на `date`. Столбец `diag_stage` пока под вопросом.
- Пропуски содержатся в столбцах: `patient_registration_start_date`,`patient_registrations_removal_causes`, `patient_death_date` и `diag_stage`. Критичность этого параметра определим позже.
- Проверка на уникальные значения и дубликаты позже.

##### RECEPTION

In [ ]:
# Знакомимся
RECEPTION.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 198923 entries, 0 to 198922
Data columns (total 10 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   DOCUMENT_ID       198923 non-null  object
 1   CCT               198923 non-null  int64 
 2   CCT_NAME          198923 non-null  object
 3   EVENT_DATE        198923 non-null  object
 4   PATIENT_ID        198923 non-null  int64 
 5   DIAGNOSIS_CODE    198923 non-null  object
 6   DIAGNOSIS_TYPE    198923 non-null  object
 7   DIAGNOSIS_STATUS  138196 non-null  object
 8   DOCUMENT_MU_ID    198923 non-null  int64 
 9   EMPLOYEE_JOB_ID   198268 non-null  object
dtypes: int64(3), object(7)
memory usage: 15.2+ MB


In [ ]:
# Визуальный осмотр
display(RECEPTION)

,DOCUMENT_ID,CCT,CCT_NAME,EVENT_DATE,PATIENT_ID,DIAGNOSIS_CODE,DIAGNOSIS_TYPE,DIAGNOSIS_STATUS,DOCUMENT_MU_ID,EMPLOYEE_JOB_ID
0,000036b2-33b7-46d1-863f-17f81febf713,14973,Осмотр эндокринолога,2023-10-16,10400713,E11.6,основной диагноз,подтвержден,179,10
1,00004908-8558-426d-b4a2-04051ccf1fcb,14973,Осмотр эндокринолога,2022-11-17,22771694,E11.9,основной диагноз,подтвержден,208,10
2,0003460c-5421-4e49-9444-221c601ee0ff,14973,Осмотр эндокринолога,2024-03-01,10616365,E11.6,основной диагноз,подтвержден,10000297,10
3,0003e9d5-d9c3-49ce-a25f-9992f9ef7e11,14973,Осмотр эндокринолога,2025-01-22,19247034,E11.8,основной диагноз,подтвержден,11321453,10
4,0005e06d-df8f-4b15-ad26-ffbdb305fb50,14973,Осмотр эндокринолога,2024-03-04,17175173,E11.7,основной диагноз,подтвержден,170,10
...,...,...,...,...,...,...,...,...,...,...
198918,fbb387a4-0903-40f2-9935-86826c362d2e,14973,Осмотр эндокринолога,2022-12-02,17345425,E11.9,основной диагноз,не подтвержден,10000351,10
198919,fbb58c39-cd8d-442d-946a-ea41b2e028f3,14973,Осмотр эндокринолога,2025-01-28,10453016,E11.7,основной диагноз,подтвержден,225,10
198920,fbb5ec7a-b4a7-42f9-a69f-e5ef660bc879,14973,Осмотр эндокринолога,2023-04-05,10755250,E11.8,основной диагноз,подтвержден,10274928,10
198921,fbb7c428-e103-4abe-aa9a-0f513d95c502,14973,Осмотр эндокринолога,2024-03-22,17903091,E11.7,основной диагноз,подтвержден,10000260,10


- Названия столбцов переводим в нижний регистр.
- Меняем тип данных в столбцах: `EVENT_DATE` на date и, возможно, в `EMPLOYEE_JOB_ID` на int.
- Пропуски есть в столбцах: `DIAGNOSIS_STATUS` и `EMPLOYEE_JOB_ID `. Неободимо изучить каждый столбец по отдельности.
- Проверка на уникальные значения и дубликаты позже.

##### LLO

In [ ]:
# Знакомимся
LLO.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 81269 entries, 0 to 81268
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   PATIENT_ID         81269 non-null  int64 
 1   ELECTRONIC_NUMBER  81269 non-null  object
 2   MNN_ID             81269 non-null  int64 
 3   DRUG_NAME_MNN      81269 non-null  object
 4   START_DATE         81269 non-null  object
 5   END_DATE           81269 non-null  object
 6   SALE_DATE          71874 non-null  object
 7   MU_ID              81269 non-null  int64 
dtypes: int64(3), object(5)
memory usage: 5.0+ MB


In [ ]:
# Визуальный осмотр
display(LLO)

,PATIENT_ID,ELECTRONIC_NUMBER,MNN_ID,DRUG_NAME_MNN,START_DATE,END_DATE,SALE_DATE,MU_ID
0,30944694,01Э4538634554,1325,Валсартан,2022-11-17,2022-12-17,NaN,10000406
1,30944694,01Э4538634731,1325,Валсартан,2022-11-17,2022-12-17,2022-11-17,10000406
2,10445134,01Э4538636350,1325,Валсартан,2022-11-17,2023-02-15,2022-11-17,134
3,18925707,01Э4538636781,9221,Бортезомиб,2022-11-17,2022-12-17,NaN,11386028
4,17077191,01Э4538637859,1325,Валсартан,2022-11-17,2022-12-17,2022-11-17,227
...,...,...,...,...,...,...,...,...
81264,10637396,01Э4559283701,333,Ибупрофен,2023-12-13,2024-01-12,NaN,10000285
81265,1901394391,01Э4559283797,12102,Абиратерон,2023-12-13,2024-01-12,2023-12-13,11708903
81266,25968595,01Э4559283813,12102,Абиратерон,2023-12-13,2024-01-12,2023-12-13,11601278
81267,10637396,01Э4559283941,333,Ибупрофен,2023-12-13,2024-01-12,2023-12-13,10000285


- Названия столбцов переводим в нижний регистр.
- Меняем тип данных в столбцах: `START_DATE`, `END_DATE` и `SALE_DATE`на date.
- Пропуски есть в столбце `SALE_DATE`. Критичность этого параметра определим позже.
- Проверка на уникальные значения и дубликаты позже.

### Предобработка данных

##### MGKR

In [ ]:
# Преобразование формата
MGKR['patient_registration_start_date'] = pd.to_datetime(MGKR['patient_registration_start_date'])
MGKR['patient_death_date'] = pd.to_datetime(MGKR['patient_death_date'])
MGKR['diag_establish_date'] = pd.to_datetime(MGKR['diag_establish_date'])

In [ ]:
# Проверка типов данных
MGKR.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 199956 entries, 0 to 199955
Data columns (total 9 columns):
 #   Column                                Non-Null Count   Dtype         
---  ------                                --------------   -----         
 0   patient_id                            199956 non-null  int64         
 1   patient_gender                        199956 non-null  object        
 2   patient_registration_status           199956 non-null  object        
 3   patient_registration_start_date       199937 non-null  datetime64[ns]
 4   patient_registrations_removal_causes  31410 non-null   object        
 5   patient_death_date                    28187 non-null   datetime64[ns]
 6   diag_code                             199956 non-null  object        
 7   diag_establish_date                   199956 non-null  datetime64[ns]
 8   diag_stage                            167200 non-null  object        
dtypes: datetime64[ns](3), int64(1), object(5)
memory usage: 13.7

In [ ]:
# Изучаем значение столбца diag_stage
MGKR['diag_stage'].value_counts(dropna=False)

diag_stage
II             53514
I              39919
NaN            32756
III            20210
IV             19977
IIIB            4947
IIA             4634
IIIA            4468
IIB             3789
IA              3301
IB              2922
неизвестно      2540
IVA             1739
IIIC            1426
IVB             1347
IA2              583
IIC              574
IA3              388
IC               361
нет данных       237
IA1              147
IVC              116
неприменимо       35
0is               19
IIIC1              3
IIIA2              2
IS                 1
0                  1
Name: count, dtype: int64

- Данный столбец для дальнеёшего анализа нам не нужен. Оставляем данные без изменений.

- Столбцы с пропусками: `patient_death_date`, `patient_registrations_removal_causes` и `patient_registration_start_date`, оставляем без изменений, так как, для дальнейшего анализа они нам не нужны.

In [ ]:
# Проверяем наличие явных дубликатов
MGKR.duplicated().sum()

np.int64(10)

In [ ]:
# Визуально остатриваем
duplicates = MGKR[MGKR.duplicated(keep=False)]  # keep=False для показа всех дублей
display(duplicates.sort_values(by=MGKR.columns.tolist()))

,patient_id,patient_gender,patient_registration_status,patient_registration_start_date,patient_registrations_removal_causes,patient_death_date,diag_code,diag_establish_date,diag_stage
43600,10309223,Мужской,Снят с учета,2006-02-02,"умер от причин, связанных с основным заболеванием",2023-02-13,C61,2005-11-18,II
43601,10309223,Мужской,Снят с учета,2006-02-02,"умер от причин, связанных с основным заболеванием",2023-02-13,C61,2005-11-18,II
114877,16598022,Женский,Стоит на учете,2000-09-11,NaN,NaT,C34.1,2018-07-02,IB
114878,16598022,Женский,Стоит на учете,2000-09-11,NaN,NaT,C34.1,2018-07-02,IB
14534,17513504,Мужской,Снят с учета,2024-04-03,"умер от причин, связанных с основным заболеванием",2025-01-22,C34.8,2024-03-20,IV
14535,17513504,Мужской,Снят с учета,2024-04-03,"умер от причин, связанных с основным заболеванием",2025-01-22,C34.8,2024-03-20,IV
44466,19927618,Мужской,Стоит на учете,2025-05-07,NaN,NaT,C34.1,2024-12-19,IV
44467,19927618,Мужской,Стоит на учете,2025-05-07,NaN,NaT,C34.1,2024-12-19,IV
40928,20228757,Мужской,Снят с учета,2024-11-22,"умер от причин, связанных с основным заболеванием",2025-01-29,C34.8,2025-01-29,NaN
40929,20228757,Мужской,Снят с учета,2024-11-22,"умер от причин, связанных с основным заболеванием",2025-01-29,C34.8,2025-01-29,NaN


In [ ]:
# Визуальный осмотр подтвердил, что строки являются полными дубликатами и не представляют ценности.
# Удаляем все, кроме первых значений
MGKR = MGKR.drop_duplicates(keep='first')

In [ ]:
# Проверяем снова
MGKR.duplicated().sum()

np.int64(0)

In [ ]:
# Проверим наличие неявных дубликатов
# Проверяем уникальные значения в столбцах и их количество
for column in ['patient_gender','patient_registration_status', 'patient_registrations_removal_causes', 'diag_code']:
    print(f'Уникальные значения в столбце {column}:')
    unique_values = MGKR[column].sort_values().unique()
    print(unique_values)
    print(f'Количество уникальных значений: {len(unique_values)}')
    print()

Уникальные значения в столбце patient_gender:
['Женский' 'Мужской']
Количество уникальных значений: 2

Уникальные значения в столбце patient_registration_status:
['Снят с учета' 'Стоит на учете']
Количество уникальных значений: 2

Уникальные значения в столбце patient_registrations_removal_causes:
['выехал' 'диагноз не подтвердился'
 'диагноз не подтвердился; диагноз не подтвердился'
 'диагноз не подтвердился; умер от другого заболевания'
 'диагноз не подтвердился; умер от причин, связанных с основным заболеванием'
 'диагноз не подтвердился; умер, причина неизвестна' 'неизвестно'
 'состоял по базалиоме' 'состоял по базалиоме; диагноз не подтвердился'
 'состоял по базалиоме; умер от другого заболевания'
 'состоял по базалиоме; умер от причин, связанных с основным заболеванием'
 'умер от другого заболевания'
 'умер от причин, связанных с основным заболеванием'
 'умер, причина неизвестна' nan]
Количество уникальных значений: 15

Уникальные значения в столбце diag_code:
['C19' 'C20' 'C21.0

- Данные выглядят корректно.

##### Датафрейм MGKR готов для объединения.

##### RECEPTION

In [ ]:
# Переводим названия всех столбцов в нижний регистр
RECEPTION.columns = RECEPTION.columns.str.lower()

In [ ]:
# Проверяем
print(RECEPTION.columns.tolist())

['document_id', 'cct', 'cct_name', 'event_date', 'patient_id', 'diagnosis_code', 'diagnosis_type', 'diagnosis_status', 'document_mu_id', 'employee_job_id']


In [ ]:
# Преобразование формата
RECEPTION['event_date'] = pd.to_datetime(RECEPTION['event_date'])

In [ ]:
# Проверяем
RECEPTION['event_date'].dtype

dtype('<M8[ns]')

- Остальные столбцы или соответсвуют форматам или для дальнейшего исследования не нужны.

- Столбцы с пропусками `DIAGNOSIS_STATUS` и `EMPLOYEE_JOB_ID `оставляем без изменений, так как, для дальнейшего анализа они нам не нужны.

In [ ]:
# Проверяем наличие явных дубликатов
RECEPTION.duplicated().sum()

np.int64(14)

In [ ]:
# Визуально остатриваем
duplicates_r = RECEPTION[RECEPTION.duplicated(keep=False)]  # keep=False для показа всех дублей
display(duplicates_r.sort_values(by=RECEPTION.columns.tolist()))

,document_id,cct,cct_name,event_date,patient_id,diagnosis_code,diagnosis_type,diagnosis_status,document_mu_id,employee_job_id
139750,17571510-ca4a-42ff-bb7d-5dc2cdd8417e,44064,Осмотр акушера-гинеколога,2025-01-16,30000016682345,C50.8,сопутствующий диагноз,NaN,11651503,4
139751,17571510-ca4a-42ff-bb7d-5dc2cdd8417e,44064,Осмотр акушера-гинеколога,2025-01-16,30000016682345,C50.8,сопутствующий диагноз,NaN,11651503,4
140856,465fdae1-85cc-4af5-954c-1f65d7c1727f,44064,Осмотр акушера-гинеколога,2025-03-04,18095969,E11.9,сопутствующий диагноз,NaN,11028303,4
140857,465fdae1-85cc-4af5-954c-1f65d7c1727f,44064,Осмотр акушера-гинеколога,2025-03-04,18095969,E11.9,сопутствующий диагноз,NaN,11028303,4
141524,5c480ff5-3bb1-4a3a-b3e8-b3d7148bcc91,44064,Осмотр акушера-гинеколога,2025-01-10,21590619,C50.4,основной диагноз,подтвержден,11747853,4
141525,5c480ff5-3bb1-4a3a-b3e8-b3d7148bcc91,44064,Осмотр акушера-гинеколога,2025-01-10,21590619,C50.4,основной диагноз,подтвержден,11747853,4
141750,66a4c8ef-fc30-4dbf-bc64-1869c4429e1f,44064,Осмотр акушера-гинеколога,2025-01-23,21590619,C50.4,основной диагноз,подтвержден,11747853,4
141751,66a4c8ef-fc30-4dbf-bc64-1869c4429e1f,44064,Осмотр акушера-гинеколога,2025-01-23,21590619,C50.4,основной диагноз,подтвержден,11747853,4
141865,6c225f84-d8a7-44e8-8d77-ab98812d909a,44064,Осмотр акушера-гинеколога,2025-01-17,20632632,C50.4,сопутствующий диагноз,NaN,11824728,4
141866,6c225f84-d8a7-44e8-8d77-ab98812d909a,44064,Осмотр акушера-гинеколога,2025-01-17,20632632,C50.4,сопутствующий диагноз,NaN,11824728,4


In [ ]:
# Визуальный осмотр подтвердил, что строки являются полными дубликатами и не представляют ценности.
# Удаляем все, кроме первых значений
RECEPTION = RECEPTION.drop_duplicates(keep='first')

In [ ]:
# Проверяем снова
RECEPTION.duplicated().sum()

np.int64(0)

In [ ]:
# Проверим наличие неявных дубликатов
# Проверяем уникальные значения в столбцах и их количество
for column in ['diagnosis_code', 'diagnosis_type', 'cct_name']:
    print(f'Уникальные значения в столбце {column}:')
    unique_values = RECEPTION[column].sort_values().unique()
    print(unique_values)
    print(f'Количество уникальных значений: {len(unique_values)}')
    print()

Уникальные значения в столбце diagnosis_code:
['C50.0' 'C50.1' 'C50.2' 'C50.3' 'C50.4' 'C50.5' 'C50.6' 'C50.8' 'C50.9'
 'C51.0' 'C51.1' 'C51.2' 'C51.8' 'C51.9' 'C52' 'C53' 'C53.0' 'C53.1'
 'C53.8' 'C53.9' 'C54' 'C54.0' 'C54.1' 'C54.2' 'C54.3' 'C54.8' 'C54.9'
 'C55' 'C56' 'C57' 'C57.0' 'C57.4' 'C57.9' 'C58' 'E10' 'E10.0' 'E10.2'
 'E10.3' 'E10.4' 'E10.5' 'E10.6' 'E10.7' 'E10.8' 'E10.9' 'E11' 'E11.0'
 'E11.1' 'E11.2' 'E11.2*' 'E11.3' 'E11.4' 'E11.4*' 'E11.5' 'E11.6' 'E11.7'
 'E11.8' 'E11.9' 'T00.0' 'T00.1' 'T00.2' 'T00.3' 'T00.6' 'T00.8' 'T00.9'
 'T01.1' 'T01.2' 'T01.3' 'T01.6' 'T01.8' 'T01.9' 'T02.1' 'T02.10' 'T02.2'
 'T02.20' 'T02.3' 'T02.30' 'T02.4' 'T02.40' 'T02.41' 'T02.5' 'T02.50'
 'T02.6' 'T02.60' 'T02.7' 'T02.70' 'T02.8' 'T02.80' 'T02.81' 'T02.9'
 'T02.90' 'T03.2' 'T03.3' 'T03.4' 'T03.8' 'T03.9' 'T05.8' 'T06.8' 'T07'
 'T08.0' 'T09.0' 'T09.1' 'T10.0' 'T11.0' 'T11.1' 'T11.6' 'T11.9' 'T13.1'
 'T13.2' 'T13.6' 'T13.8' 'T13.9' 'T14' 'T14.0' 'T14.1' 'T14.2' 'T14.3'
 'T14.4' 'T14.5' 'T14.

- Мы наблюдаем интересное значение в столбце `cct_name`, а именно: `Осмотр нефролога`.

In [ ]:
# Посчитаем количесво значений связанных с неврологом
target_values = [
    'Осмотр невролога',
    'Осмотр невролога (1-17 лет)',
    'Осмотр нефролога'
]
# Запустим цикл
for value in target_values:
    count = (RECEPTION['cct_name'] == value).sum()
    print(f"{value}: {count} строк")

Осмотр невролога: 1298 строк
Осмотр невролога (1-17 лет): 3 строк
Осмотр нефролога: 594 строк


- Не похоже на ошибку, слишком большое количество значений, 594. Следовательно, такой врач существует(для меня это стало открытием). В любом случае, этот столбец для дальнейших исследований нам не нужен.

##### Датафрейм RECEPTION готов для объединения.

##### LLO

In [ ]:
# Переводим названия всех столбцов в нижний регистр
LLO.columns = LLO.columns.str.lower()

In [ ]:
# Преобразование формата
LLO['start_date'] = pd.to_datetime(LLO['start_date'])
LLO['end_date'] = pd.to_datetime(LLO['end_date'])
LLO['sale_date'] = pd.to_datetime(LLO['sale_date'])

In [ ]:
# Проверка преобразований
LLO.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 81269 entries, 0 to 81268
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   patient_id         81269 non-null  int64         
 1   electronic_number  81269 non-null  object        
 2   mnn_id             81269 non-null  int64         
 3   drug_name_mnn      81269 non-null  object        
 4   start_date         81269 non-null  datetime64[ns]
 5   end_date           81269 non-null  datetime64[ns]
 6   sale_date          71874 non-null  datetime64[ns]
 7   mu_id              81269 non-null  int64         
dtypes: datetime64[ns](3), int64(3), object(2)
memory usage: 5.0+ MB


- Пропуски в столбце `sale_date` оставляем без изменений, так как, для дальнейшего анализа он нам не нужен.

In [ ]:
# Проверяем наличие явных дубликатов
LLO.duplicated().sum()

np.int64(0)

In [ ]:
# Проверим наличие неявных дубликатов
# Проверяем уникальные значения в столбце и их количество
unique_values = LLO['drug_name_mnn'].sort_values().unique()
print(f'Уникальные значения ({len(unique_values)}):\n{unique_values}')

Уникальные значения (14):
['Абатацепт' 'Абиратерон' 'Азеластин+Мометазон'
 'Аклидиния бромид+Формотерол' 'Ацикловир' 'Бозентан' 'Бортезомиб'
 'Валсартан' 'Железа III гидроксид декстран' 'Ибупрофен' 'Инсулин аспарт'
 'Фторметолон' 'Цитиколин' 'Эксеместан']


- Данные корректны.

##### Датафрейм LLO готов для объединения.

### Объединение в один датафрейм

В финальной таблице нам нужны:
- из MGKR поля: `diag_code`, `diag_establish_date`
- из LLO поля: `drug_name_mnn`, `mu_id` , `start_date_first`, `start_date_last`
- из RECEPTION поля: `diagnosis_code`, `event_date`.

Объединяем по `patient_id`.

За основу берём таблицу MGKR к которой присоединяем(left join) совподающие `patient_id` из других таблиц.

In [ ]:
df = MGKR.merge(LLO, on='patient_id').merge(RECEPTION, on='patient_id')

In [ ]:
# Проверяем
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 289586 entries, 0 to 289585
Data columns (total 25 columns):
 #   Column                                Non-Null Count   Dtype         
---  ------                                --------------   -----         
 0   patient_id                            289586 non-null  int64         
 1   patient_gender                        289586 non-null  object        
 2   patient_registration_status           289586 non-null  object        
 3   patient_registration_start_date       289586 non-null  datetime64[ns]
 4   patient_registrations_removal_causes  25754 non-null   object        
 5   patient_death_date                    22613 non-null   datetime64[ns]
 6   diag_code                             289586 non-null  object        
 7   diag_establish_date                   289586 non-null  datetime64[ns]
 8   diag_stage                            255270 non-null  object        
 9   electronic_number                     289586 non-null  obje

In [ ]:
# Визуально осматриваем
display(df)

,patient_id,patient_gender,patient_registration_status,patient_registration_start_date,patient_registrations_removal_causes,patient_death_date,diag_code,diag_establish_date,diag_stage,electronic_number,...,mu_id,document_id,cct,cct_name,event_date,diagnosis_code,diagnosis_type,diagnosis_status,document_mu_id,employee_job_id
0,10592172,Мужской,Стоит на учете,2017-04-10,NaN,NaT,C61,2017-04-10,II,01Э4539852980,...,10000355,a56715f4-b2f8-4805-a088-fcb3820c2d34,14974,Осмотр хирурга,2023-05-18,E11.7,основной диагноз,не подтвержден,10000355,3
1,10592172,Мужской,Стоит на учете,2017-04-10,NaN,NaT,C61,2017-04-10,II,01Э4539852980,...,10000355,1eeb3109-f992-404b-9f12-945d99f3032a,14973,Осмотр эндокринолога,2023-01-25,E11.7,основной диагноз,подтвержден,10000355,10
2,10592172,Мужской,Стоит на учете,2017-04-10,NaN,NaT,C61,2017-04-10,II,01Э4539852980,...,10000355,4f72a960-0205-4b55-a250-5cc752d47334,14973,Осмотр эндокринолога,2024-11-05,E11.7,основной диагноз,подтвержден,10000355,10
3,10592172,Мужской,Стоит на учете,2017-04-10,NaN,NaT,C61,2017-04-10,II,01Э4539852980,...,10000355,5ec31264-9cd3-4ae4-b808-15d5e9a0de9d,14973,Осмотр эндокринолога,2022-12-27,E11.7,основной диагноз,подтвержден,10000355,10
4,10592172,Мужской,Стоит на учете,2017-04-10,NaN,NaT,C61,2017-04-10,II,01Э4537946405,...,10000355,a56715f4-b2f8-4805-a088-fcb3820c2d34,14974,Осмотр хирурга,2023-05-18,E11.7,основной диагноз,не подтвержден,10000355,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
289581,23962312,Мужской,Стоит на учете,2023-03-16,NaN,NaT,C61,2023-02-10,II,01Э4558468169,...,231,90bc8933-ae69-4a72-a56f-6305f409f367,90060,Осмотр врача-онкоуролога,2024-02-29,E11.8,сопутствующий диагноз,NaN,11601278,26
289582,23962312,Мужской,Стоит на учете,2023-03-16,NaN,NaT,C61,2023-02-10,II,01Э4558468169,...,231,93abe96b-3ae8-4c36-937b-ac5ef2b1015f,90060,Осмотр врача-онкоуролога,2024-03-22,E11.8,сопутствующий диагноз,NaN,11601278,26
289583,23962312,Мужской,Стоит на учете,2023-03-16,NaN,NaT,C61,2023-02-10,II,01Э4558468169,...,231,953b608c-a555-4706-be9f-01fb767a7e39,90060,Осмотр врача-онкоуролога,2023-03-31,E11.8,сопутствующий диагноз,NaN,11601278,26
289584,23962312,Мужской,Стоит на учете,2023-03-16,NaN,NaT,C61,2023-02-10,II,01Э4558468169,...,231,e93957c5-4f4e-4e4b-9bcb-6fb29b3e6f6c,90060,Осмотр врача-онкоуролога,2023-12-05,E11.8,сопутствующий диагноз,NaN,11601278,26


### Задача 1 Решение

Логика:

- Сначала создаем CTE для пациентов, которые впервые получали Абиратерон в 2023-2024 годах, с учетом всех условий.
- Затем создаем CTE для вычисления дат первого и последнего рецептов с получением лекарства для каждого пациента.
- В основном запросе объединяем эти данные, выбирая только первую запись для каждого пациента (где rn = 1).

In [ ]:
# Создаём подключение
conn = sqlite3.connect(':memory:')
df.to_sql('df', conn, index=False, if_exists='replace')

# Финальный SQL-запрос
query = """
-- Создаем CTE для идентификации пациентов, впервые получавших Абиратерон в 2023 или 2024
WITH FirstAbirateronePatients AS (
    SELECT
        patient_id, -- ID пациента
        diag_code, -- Код диагноза
        diag_establish_date, -- Дата установления диагноза
        drug_name_mnn, -- Название препарата
        mu_id, -- ID медицинского учреждения
        start_date, -- Дата назначения рецепта
        sale_date, -- Дата получения препарата
        ROW_NUMBER() OVER (PARTITION BY patient_id ORDER BY start_date) as rn -- Нумеруем записи для каждого пациента по дате назначения, чтобы потом выбрать только первую запись (rn = 1)
    FROM df
    WHERE
        patient_registration_status = 'Стоит на учете' -- Только пациенты на учете
        AND diag_code = 'C61' -- С диагнозом C61
        AND drug_name_mnn = 'Абиратерон' -- Получавшие Абиратерон
        AND strftime('%Y', start_date) IN ('2023', '2024') -- Рецепты только за 2023 или 2024
        AND sale_date IS NOT NULL -- Фильтруем по фактическому получению
),
-- Создаем CTE для вычисления дат первого и последнего рецептов
PatientFirstLastDates AS (
    SELECT
        patient_id,
        MIN(CASE WHEN sale_date IS NOT NULL THEN start_date END) as first_prescription_date, -- Первая дата рецепта с получением препарата
        MAX(CASE WHEN sale_date IS NOT NULL THEN start_date END) as last_prescription_date -- Последняя дата рецепта с получением препарата
    FROM df
    WHERE
        patient_registration_status = 'Стоит на учете'
        AND diag_code = 'C61'
        AND drug_name_mnn = 'Абиратерон'
        AND strftime('%Y', start_date) IN ('2023', '2024')
    GROUP BY patient_id -- Группируем по пациенту, чтобы получить одну запись
)
-- Основной запрос, объединяющий данные из обоих CTE
SELECT
    f.patient_id as "ИД пациента",
    f.diag_code as "Диагноз в МГКР",
    f.diag_establish_date as "Дата постановки диагноза в МГКР",
    f.drug_name_mnn as "Наименование ЛП",
    f.mu_id as "Медицинская организация",
    p.first_prescription_date as "Дата первого выписанного рецепта c получением ЛП",
    p.last_prescription_date as "Дата последнего выписанного рецепта с получением ЛП"
FROM FirstAbirateronePatients as f
JOIN PatientFirstLastDates as p on f.patient_id = p.patient_id
WHERE f.rn = 1 -- Берем только первую запись для каждого пациента (rn = 1);
"""
result = pd.read_sql_query(query, conn)
print("\nРезультаты:")
display(result)

conn.close()


Результаты:


,ИД пациента,Диагноз в МГКР,Дата постановки диагноза в МГКР,Наименование ЛП,Медицинская организация,Дата первого выписанного рецепта c получением ЛП,Дата последнего выписанного рецепта с получением ЛП
0,10667562,C61,2013-04-24 00:00:00,Абиратерон,12032803,2024-11-27 00:00:00,2024-12-26 00:00:00
1,10945654,C61,2017-01-18 00:00:00,Абиратерон,11708903,2023-02-03 00:00:00,2024-07-08 00:00:00
2,15988349,C61,2017-07-01 00:00:00,Абиратерон,11708903,2023-01-03 00:00:00,2024-10-16 00:00:00
3,16000924,C61,2018-02-28 00:00:00,Абиратерон,11394228,2023-02-07 00:00:00,2023-08-02 00:00:00
4,16292041,C61,2020-06-25 00:00:00,Абиратерон,11708903,2024-04-02 00:00:00,2024-12-05 00:00:00
...,...,...,...,...,...,...,...
103,30000004517883,C61,2023-05-30 00:00:00,Абиратерон,11708903,2023-07-24 00:00:00,2024-12-21 00:00:00
104,30000008416139,C61,2018-04-05 00:00:00,Абиратерон,11381928,2023-01-25 00:00:00,2024-10-16 00:00:00
105,30000014213905,C61,2023-05-22 00:00:00,Абиратерон,11708903,2023-06-27 00:00:00,2024-12-15 00:00:00
106,30000014219113,C61,2023-06-11 00:00:00,Абиратерон,11708903,2023-10-20 00:00:00,2024-12-20 00:00:00


### Задача 2 Решение

Логика:

- Добавляем CTE, в котором группирует данные по patient_id
- Добавляем флаг и дату с последним подтверждением
- Фильтруем по '%E11%' и по подтверждению
- Добавлем результаты в финальный запрос

In [ ]:
# Создаём подключение
conn = sqlite3.connect(':memory:')
df.to_sql('df', conn, index=False, if_exists='replace')

# Финальный SQL-запрос
query = """
-- Создаем CTE для идентификации пациентов, впервые получавших Абиратерон в 2023 или 2024
WITH FirstAbirateronePatients AS (
    SELECT
        patient_id, -- ID пациента
        diag_code, -- Код диагноза
        diag_establish_date, -- Дата установления диагноза
        drug_name_mnn, -- Название препарата
        mu_id, -- ID медицинского учреждения
        start_date, -- Дата назначения рецепта
        sale_date, -- Дата получения препарата
        ROW_NUMBER() OVER (PARTITION BY patient_id ORDER BY start_date) as rn -- Нумеруем записи для каждого пациента по дате назначения, чтобы потом выбрать только первую запись (rn = 1)
    FROM df
    WHERE
        patient_registration_status = 'Стоит на учете' -- Только пациенты на учете
        AND diag_code = 'C61' -- С диагнозом C61
        AND drug_name_mnn = 'Абиратерон' -- Получавшие Абиратерон
        AND strftime('%Y', start_date) IN ('2023', '2024') -- Рецепты только за 2023 или 2024
        AND sale_date IS NOT NULL -- Фильтруем по фактическому получению
),
-- Создаем CTE для вычисления дат первого и последнего рецептов
PatientFirstLastDates AS (
    SELECT
        patient_id,
        MIN(CASE WHEN sale_date IS NOT NULL THEN start_date END) as first_prescription_date, -- Первая дата рецепта с получением препарата
        MAX(CASE WHEN sale_date IS NOT NULL THEN start_date END) as last_prescription_date -- Последняя дата рецепта с получением препарата
    FROM df
    WHERE
        patient_registration_status = 'Стоит на учете'
        AND diag_code = 'C61'
        AND drug_name_mnn = 'Абиратерон'
        AND strftime('%Y', start_date) IN ('2023', '2024')
    GROUP BY patient_id -- Группируем по пациенту, чтобы получить одну запись
),
-- Новое CTE для информации о диагнозе E11
PatientE11Info AS (
    SELECT
        patient_id,
        MAX(CASE WHEN diagnosis_code LIKE '%E11%' AND diagnosis_status = 'подтвержден' THEN 1 ELSE 0 END) as has_e11_diagnosis, -- Флаг наличия подтвержденного E11
        MAX(CASE WHEN diagnosis_code LIKE '%E11%' AND diagnosis_status = 'подтвержден' THEN event_date END) as last_e11_date -- Дата последнего подтвержденного E11
    FROM df
    WHERE
        diagnosis_code LIKE '%E11%' -- Диагнозы включающие в себя E11
        AND diagnosis_status = 'подтвержден'
    GROUP BY patient_id
)
-- Основной запрос, объединяющий данные из трёх CTE
SELECT
    f.patient_id as "ИД пациента",
    f.diag_code as "Диагноз в МГКР",
    f.diag_establish_date as "Дата постановки диагноза в МГКР",
    f.drug_name_mnn as "Наименование ЛП",
    f.mu_id as "Медицинская организация",
    p.first_prescription_date as "Дата первого выписанного рецепта c получением ЛП",
    p.last_prescription_date as "Дата последнего выписанного рецепта с получением ЛП",
    -- Новые столбцы:
    CASE WHEN e.has_e11_diagnosis = 1 THEN 'да' ELSE 'нет' END as "Наличие приема с подтвержденным диагнозом E11",
    e.last_e11_date as "Дата последнего приема с подтвержденным диагнозом E11"
FROM FirstAbirateronePatients as f
JOIN PatientFirstLastDates as p on f.patient_id = p.patient_id
LEFT JOIN PatientE11Info as e on f.patient_id = e.patient_id -- left так как не у всех пациентов может быть диагноз E11
WHERE f.rn = 1;
"""
result = pd.read_sql_query(query, conn)

# Снимаем ограничение на максимальное количество строк для вывода
pd.set_option('display.max_rows', None)

print("\nРезультаты:")
display(result)

conn.close()


Результаты:


,ИД пациента,Диагноз в МГКР,Дата постановки диагноза в МГКР,Наименование ЛП,Медицинская организация,Дата первого выписанного рецепта c получением ЛП,Дата последнего выписанного рецепта с получением ЛП,Наличие приема с подтвержденным диагнозом E11,Дата последнего приема с подтвержденным диагнозом E11
0,10667562,C61,2013-04-24 00:00:00,Абиратерон,12032803,2024-11-27 00:00:00,2024-12-26 00:00:00,да,2024-10-31 00:00:00
1,10945654,C61,2017-01-18 00:00:00,Абиратерон,11708903,2023-02-03 00:00:00,2024-07-08 00:00:00,да,2024-12-07 00:00:00
2,15988349,C61,2017-07-01 00:00:00,Абиратерон,11708903,2023-01-03 00:00:00,2024-10-16 00:00:00,да,2024-06-28 00:00:00
3,16000924,C61,2018-02-28 00:00:00,Абиратерон,11394228,2023-02-07 00:00:00,2023-08-02 00:00:00,да,2025-05-07 00:00:00
4,16292041,C61,2020-06-25 00:00:00,Абиратерон,11708903,2024-04-02 00:00:00,2024-12-05 00:00:00,да,2024-07-10 00:00:00
5,16368892,C61,2021-12-01 00:00:00,Абиратерон,11708903,2024-04-10 00:00:00,2024-08-30 00:00:00,нет,None
6,16410711,C61,2012-11-01 00:00:00,Абиратерон,10330308,2023-01-26 00:00:00,2024-12-28 00:00:00,да,2024-10-09 00:00:00
7,16480578,C61,2018-07-03 00:00:00,Абиратерон,10330308,2023-01-05 00:00:00,2024-12-09 00:00:00,да,2024-10-25 00:00:00
8,16695977,C61,2011-07-29 00:00:00,Абиратерон,12032803,2024-10-07 00:00:00,2024-12-04 00:00:00,нет,None
9,16790279,C61,2010-01-31 00:00:00,Абиратерон,11708903,2023-03-06 00:00:00,2024-12-28 00:00:00,да,2025-07-01 00:00:00
